# Import Modules

In [1]:
import torch

import sys
sys.path.append('..')

from gpt import GPTLanguageModel, generate, get_device
from lora import add_lora

# Load the Pre-Trained Model
Along with the token mapping

In [2]:
CHECKPOINTS_PATH = '../checkpoints'
BASE_PATH = f'{CHECKPOINTS_PATH}/gpt_shakes_best.pt'
LORA_PATH = f'{CHECKPOINTS_PATH}/lorat1_gpt_shakes_best.pt'

RANDOM_SEED = 42

device = get_device()

checkpoints = torch.load(BASE_PATH, map_location=device, weights_only=False)
config = checkpoints['config']

model = GPTLanguageModel(
    vocab_size=config['vocab_size'],
    n_embd=config['n_embd'],
    n_head=config['n_head'],
    block_size=config['block_size'],
    n_layer=config['n_layer'],
    dropout=config['dropout'],
).to(device)

model.load_state_dict(checkpoints['model_state_dict'])

stoi = config['stoi']
itos = config['itos']

encode = lambda s: [stoi[ch] for ch in s]
decode = lambda l: ''.join(itos[i] for i in l)

Check the output of the original model:

In [3]:
torch.manual_seed(RANDOM_SEED)

model.eval()
print(generate(
        model, encode, decode, device, 
        prompt='\n', max_new_tokens=500))


LUCIO:
Bring you, and come to glorious prick.

TYBALT:
My lord, you cannot speak of your prince's grounds.

FLORIZEL:
This no more. I warrant the east any your majesty.

LEONTES:
Good man, why do, that not the gods myself?

MARCIUS:
If the duke have seen to be so heir, I should say.

LEONTES:
Good man!
Then, be not.

First Officer:
Look, 'tis some that, you may.
I do not lose fear me by: you why I say,
I shall do; for I shall report him.

First Lady:
He does are none, our father even yourselves.


# Load LoRA
1. Load the base model first
2. Add LoRA layers, just like during training.
3. Then load the trained LoRA dict

In [4]:
# Load pretrained model
lora_model = GPTLanguageModel(
    vocab_size=config['vocab_size'],
    n_embd=config['n_embd'],
    n_head=config['n_head'],
    block_size=config['block_size'],
    n_layer=config['n_layer'],
    dropout=config['dropout'],
).to(device)

lora_model.load_state_dict(checkpoints['model_state_dict'])

# Add LoRA layers
add_lora(lora_model, rank=8, alpha=16, targets=('query', 'value'))
lora_model = lora_model.to(device)

Added LoRA layers here but haven't loaded trained LoRA yet.

So here output should be the same as the original model:

In [5]:
torch.manual_seed(RANDOM_SEED)

lora_model.eval()
print(generate(
        lora_model, encode, decode, device, 
        prompt='\n', max_new_tokens=500))


LUCIO:
Bring you, and come to glorious prick.

TYBALT:
My lord, you cannot speak of your prince's grounds.

FLORIZEL:
This no more. I warrant the east any your majesty.

LEONTES:
Good man, why do, that not the gods myself?

MARCIUS:
If the duke have seen to be so heir, I should say.

LEONTES:
Good man!
Then, be not.

First Officer:
Look, 'tis some that, you may.
I do not lose fear me by: you why I say,
I shall do; for I shall report him.

First Lady:
He does are none, our father even yourselves.


Loading returns a `_IncompatibleKeys`, a named tuple with two fields:
* `missing_keys`: list of strings. Paramter names in the model but not in the input `state_dict`
* `unexpected_keys`: list of strings. Parameter names in the input `state_dict` but not in the model. Should be empty here.

In [6]:
# Load LoRA
lora_state = torch.load(LORA_PATH, map_location=device)
missing, unexpected = lora_model.load_state_dict(lora_state, strict=False)
print(missing)
print(unexpected)       # Must be empty

['token_embedding_table.weight', 'position_embedding_table.weight', 'blocks.0.sa.heads.0.tril', 'blocks.0.sa.heads.0.query.linear.weight', 'blocks.0.sa.heads.0.key.weight', 'blocks.0.sa.heads.0.value.linear.weight', 'blocks.0.sa.heads.1.tril', 'blocks.0.sa.heads.1.query.linear.weight', 'blocks.0.sa.heads.1.key.weight', 'blocks.0.sa.heads.1.value.linear.weight', 'blocks.0.sa.heads.2.tril', 'blocks.0.sa.heads.2.query.linear.weight', 'blocks.0.sa.heads.2.key.weight', 'blocks.0.sa.heads.2.value.linear.weight', 'blocks.0.sa.heads.3.tril', 'blocks.0.sa.heads.3.query.linear.weight', 'blocks.0.sa.heads.3.key.weight', 'blocks.0.sa.heads.3.value.linear.weight', 'blocks.0.sa.heads.4.tril', 'blocks.0.sa.heads.4.query.linear.weight', 'blocks.0.sa.heads.4.key.weight', 'blocks.0.sa.heads.4.value.linear.weight', 'blocks.0.sa.heads.5.tril', 'blocks.0.sa.heads.5.query.linear.weight', 'blocks.0.sa.heads.5.key.weight', 'blocks.0.sa.heads.5.value.linear.weight', 'blocks.0.sa.proj.weight', 'blocks.0.sa.proj

The output now changes to "Sherlock" style:

In [7]:
torch.manual_seed(RANDOM_SEED)

lora_model.eval()
print(generate(
        lora_model, encode, decode, device, 
        prompt='\n', max_new_tokens=500))


-

Lady riched what is as I wonder that the cabmats. My mind, they was him,
I am a nept-perdicical to meet, and one of the northing of Some expresence
anyon alreach to Nentry Vanspergarent; The room and a command guess,
permissed the Darst. There are you avoident ill you saw you.

John Ferrier rain souls, she appeared to alive a month.
There is it that I saw the staggarden to fear a mercy to make
the rich of a death constonation. The best good had not no new arrand forfeit no was
man to that me 


# Comparison

In [9]:
prompt = 'To be or not to be, '
max_new_tokens = 500

print('[BASE (Tiny Shakespeare)]')
print(generate(model, encode, decode, device, 
            prompt=prompt, max_new_tokens=max_new_tokens))

print('\n-------------------------')

print('[BASE + LoRA (Conan Doyle)]')
print(generate(lora_model, encode, decode, device, 
            prompt=prompt, max_new_tokens=max_new_tokens))


[BASE (Tiny Shakespeare)]
To be or not to be, my voice.

WARWICK:
How now should you wept, speak forthly father?
Whom love is Clarence? all the sun an hour-spirit?
She can yield full of courteous love and open!
Farewell, so I hope as long the foot in destroy
Of Berkeley, pursuing Gloucester. What says you,
To war hours, and it sense you our usurpress,
Access to every the traitor's deed?

HASTINGS:
Away, and but already of you
Both we are good coming and his friendship
Looks to your king, from the fearful cause of his roud!

KING RICHARD II:

-------------------------
[BASE + LoRA (Conan Doyle)]
To be or not to be, sir, farewell singly
against a complain without war on their fexasing eagle and putfial. Someton
passed to out. The ring revidence and matters at the first of this
masters in ha jewel, and beard off-ran devour, and I'll showing it them. How
doctors and you fury preated loss drink to the volument of which were
about the moment.

At the queen in his third other. Good faith, the